# 03 - Section, Horizon, and Classical Model Experiments

Uses the pre-aggregated filing-level table so we do not reprocess 1.3M sentences for every experiment.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report
)
import joblib

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "processed" / "filing_level.parquet"
INTERIM_DIR = ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
df = pd.read_parquet(DATA_PATH)
print("Before min-sentence filter:", df.shape)
print(df.groupby(["section_name", "split"]).size().unstack(fill_value=0))

df = df[df["n_sentences"] >= 10].reset_index(drop=True)
print("\nAfter n_sentences >= 10:", df.shape)
print(df.groupby(["section_name", "split"]).size().unstack(fill_value=0))
for h in ["1d", "5d", "30d"]:
    print(h, df[f"label_{h}"].value_counts(normalize=True).round(3).to_dict())

Before min-sentence filter: (6295, 14)
split         test  train  validate
section_name                       
1              110    917        39
1A             111    665        42
2              111    933        39
7              111    951        41
8              112    940        42
ALL            112    977        42

After n_sentences >= 10: (5047, 14)
split         test  train  validate
section_name                       
1              109    887        39
1A              92    623        40
2               26    406        10
7              111    792        41
8               91    626        23
ALL            112    977        42
1d {1: 0.522, 0: 0.478}
5d {1: 0.525, 0: 0.475}
30d {0: 0.529, 1: 0.471}


In [3]:
def metrics_dict(y_true, y_pred, y_score):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score),
    }

def fit_predict(pipe, train_df, eval_df, label_col):
    pipe.fit(train_df["text"], train_df[label_col])
    y_pred = pipe.predict(eval_df["text"])
    if hasattr(pipe, "predict_proba"):
        y_score = pipe.predict_proba(eval_df["text"])[:, 1]
    else:
        y_score = pipe.decision_function(eval_df["text"])
    return y_pred, y_score

def evaluate(pipe, train_df, eval_df, label_col):
    y_pred, y_score = fit_predict(pipe, train_df, eval_df, label_col)
    metrics = metrics_dict(eval_df[label_col], y_pred, y_score)
    metrics.update({"n_train": len(train_df), "n_eval": len(eval_df)})
    return metrics

In [4]:
def tfidf(**overrides):
    params = dict(
        max_features=10000, ngram_range=(1, 2), stop_words="english", min_df=2
    )
    params.update(overrides)
    return TfidfVectorizer(**params)

def make_model(kind):
    improved_tfidf = dict(max_features=20000, min_df=3, max_df=0.9, sublinear_tf=True)
    if kind == "base_logreg":
        return Pipeline([
            ("tfidf", tfidf()),
            ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
        ])
    if kind == "improved_logreg":
        return Pipeline([
            ("tfidf", tfidf(**improved_tfidf)),
            ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
        ])
    if kind == "unbalanced_logreg":
        return Pipeline([
            ("tfidf", tfidf(**improved_tfidf)),
            ("clf", LogisticRegression(max_iter=1000, random_state=42)),
        ])
    if kind == "complement_nb":
        return Pipeline([
            ("tfidf", tfidf(**improved_tfidf, norm=None)),
            ("clf", ComplementNB()),
        ])
    if kind == "linearsvc":
        return Pipeline([
            ("tfidf", tfidf(**improved_tfidf)),
            ("clf", LinearSVC(class_weight="balanced", max_iter=5000, random_state=42)),
        ])
    raise ValueError(kind)

MODEL_KINDS = [
    "base_logreg", "improved_logreg", "unbalanced_logreg", "complement_nb", "linearsvc"
]

## 1. Classical model comparison on ALL sections / 30d

In [5]:
section = "ALL"
horizon = "30d"
label_col = f"label_{horizon}"

train_sec = df[(df.section_name == section) & (df.split == "train")]
val_sec = df[(df.section_name == section) & (df.split == "validate")]
test_sec = df[(df.section_name == section) & (df.split == "test")]

rows = []
for kind in MODEL_KINDS:
    for split_name, eval_df in [("validation", val_sec), ("test", test_sec)]:
        m = evaluate(make_model(kind), train_sec, eval_df, label_col)
        rows.append({"model": kind, "section": section, "horizon": horizon, "eval_split": split_name, **m})

model_results = pd.DataFrame(rows)
model_results.to_csv(INTERIM_DIR / "classical_model_results.csv", index=False)
print(model_results.sort_values(["eval_split", "pr_auc"], ascending=[True, False]).to_string(index=False))

best_kind = (
    model_results[model_results.eval_split == "validation"]
.sort_values("pr_auc", ascending=False)
.iloc[0]["model"]
)
print("\nBest model by validation PR-AUC:", best_kind)

            model section horizon eval_split  accuracy  precision   recall       f1  roc_auc   pr_auc  n_train  n_eval
        linearsvc     ALL     30d       test  0.544643   0.562500 0.610169 0.585366 0.581388 0.642878      977     112
  improved_logreg     ALL     30d       test  0.553571   0.569231 0.627119 0.596774 0.587784 0.630556      977     112
unbalanced_logreg     ALL     30d       test  0.589286   0.724138 0.355932 0.477273 0.589383 0.630097      977     112
      base_logreg     ALL     30d       test  0.562500   0.578125 0.627119 0.601626 0.585865 0.616633      977     112
    complement_nb     ALL     30d       test  0.562500   0.569444 0.694915 0.625954 0.547490 0.561311      977     112
      base_logreg     ALL     30d validation  0.452381   0.523810 0.458333 0.488889 0.483796 0.614578      977      42
unbalanced_logreg     ALL     30d validation  0.428571   0.500000 0.291667 0.368421 0.469907 0.609419      977      42
  improved_logreg     ALL     30d validation  0.

## 2. Section ablation on 30d

In [6]:
section_rows = []
for sec in ["ALL", "1A", "7", "8", "1", "2"]:
    tr = df[(df.section_name == sec) & (df.split == "train")]
    va = df[(df.section_name == sec) & (df.split == "validate")]
    te = df[(df.section_name == sec) & (df.split == "test")]
    if len(tr) < 10 or len(te) < 5:
        continue
    for split_name, eval_df in [("validation", va), ("test", te)]:
        m = evaluate(make_model(best_kind), tr, eval_df, "label_30d")
        section_rows.append({"section": sec, "eval_split": split_name, **m})

section_results = pd.DataFrame(section_rows)
section_results.to_csv(INTERIM_DIR / "section_results_30d.csv", index=False)
print(section_results[section_results.eval_split == "test"].sort_values("pr_auc", ascending=False).to_string(index=False))

val_section_leaders = (
    section_results[(section_results.eval_split == "validation") & (section_results.n_eval >= 20)]
    .sort_values("pr_auc", ascending=False)
)
best_section = val_section_leaders.iloc[0]["section"] if len(val_section_leaders) else "ALL"
horizon_sections = val_section_leaders["section"].head(4).tolist()
if "ALL" not in horizon_sections:
    horizon_sections = ["ALL"] + horizon_sections[:1]
print("\nValidation section leaders (n_eval >= 20):")
print(val_section_leaders[["section", "pr_auc", "f1", "n_eval"]].to_string(index=False))
print("Best section by validation PR-AUC:", best_section)
print("Sections used for horizon comparison:", horizon_sections)

section eval_split  accuracy  precision   recall       f1  roc_auc   pr_auc  n_train  n_eval
      7       test  0.630631   0.660377 0.603448 0.630631 0.640208 0.655669      792     111
    ALL       test  0.562500   0.578125 0.627119 0.601626 0.585865 0.616633      977     112
      8       test  0.516484   0.558824 0.395833 0.463415 0.520349 0.583998      626      91
      1       test  0.550459   0.600000 0.421053 0.494845 0.581646 0.583173      887     109
     1A       test  0.543478   0.558140 0.510638 0.533333 0.576832 0.581119      623      92
      2       test  0.500000   0.545455 0.428571 0.480000 0.482143 0.522326      406      26

Validation section leaders (n_eval >= 20):
section   pr_auc       f1  n_eval
    ALL 0.614578 0.488889      42
     1A 0.549897 0.595745      40
      7 0.541967 0.358974      41
      1 0.536811 0.368421      39
      8 0.412338 0.320000      23
Best section by validation PR-AUC: ALL
Sections used for horizon comparison: ['ALL', '1A', '7', '1']


## 3. Horizon comparison (1d / 5d / 30d)

In [7]:
horizon_rows = []
for sec in horizon_sections:
    for horizon in ["1d", "5d", "30d"]:
        label_col = f"label_{horizon}"
        tr = df[(df.section_name == sec) & (df.split == "train")]
        va = df[(df.section_name == sec) & (df.split == "validate")]
        te = df[(df.section_name == sec) & (df.split == "test")]
        for split_name, eval_df in [("validation", va), ("test", te)]:
            m = evaluate(make_model(best_kind), tr, eval_df, label_col)
            horizon_rows.append({
                "section": sec, "horizon": horizon, "eval_split": split_name, **m
            })

horizon_results = pd.DataFrame(horizon_rows)
horizon_results.to_csv(INTERIM_DIR / "horizon_results.csv", index=False)
print(horizon_results[horizon_results.eval_split == "test"].sort_values(["section", "horizon"]).to_string(index=False))

section horizon eval_split  accuracy  precision   recall       f1  roc_auc   pr_auc  n_train  n_eval
      1      1d       test  0.495413   0.642857 0.285714 0.395604 0.497239 0.592958      887     109
      1     30d       test  0.550459   0.600000 0.421053 0.494845 0.581646 0.583173      887     109
      1      5d       test  0.477064   0.633333 0.292308 0.400000 0.543706 0.634904      887     109
     1A      1d       test  0.510870   0.553191 0.520000 0.536082 0.479048 0.544009      623      92
     1A     30d       test  0.543478   0.558140 0.510638 0.533333 0.576832 0.581119      623      92
     1A      5d       test  0.510870   0.546667 0.788462 0.645669 0.515865 0.587622      623      92
      7      1d       test  0.531532   0.642857 0.421875 0.509434 0.523271 0.652043      792     111
      7     30d       test  0.630631   0.660377 0.603448 0.630631 0.640208 0.655669      792     111
      7      5d       test  0.540541   0.674419 0.439394 0.532110 0.557912 0.638984      79

## 4. Summary

In [8]:
print("Best classical model:", best_kind)
print("Best 30d section (validation PR-AUC, n_eval >= 20):", best_section)
print("Horizon sections:", horizon_sections)

print("\nValidation leaders by PR-AUC (n_eval >= 20):")
model_val = model_results[(model_results.eval_split == "validation") & (model_results.n_eval >= 20)]
section_val = section_results[(section_results.eval_split == "validation") & (section_results.n_eval >= 20)]
horizon_val = horizon_results[(horizon_results.eval_split == "validation") & (horizon_results.n_eval >= 20)]

leaders = pd.concat([
    model_val.assign(experiment="model", section="ALL", horizon="30d"),
    section_val.assign(experiment="section", model=best_kind, horizon="30d"),
    horizon_val.assign(experiment="horizon", model=best_kind),
], ignore_index=True)
print(leaders.sort_values("pr_auc", ascending=False).head(10).to_string(index=False))

print("\nSaved:")
print(INTERIM_DIR / "classical_model_results.csv")
print(INTERIM_DIR / "section_results_30d.csv")
print(INTERIM_DIR / "horizon_results.csv")

Best classical model: base_logreg
Best 30d section (validation PR-AUC, n_eval >= 20): ALL
Horizon sections: ['ALL', '1A', '7', '1']

Validation leaders by PR-AUC (n_eval >= 20):
            model section horizon eval_split  accuracy  precision   recall       f1  roc_auc   pr_auc  n_train  n_eval experiment
      base_logreg       7      1d validation  0.512195   0.571429 0.190476 0.285714 0.616667 0.679140      792      41    horizon
      base_logreg       7      5d validation  0.414634   0.538462 0.280000 0.368421 0.537500 0.645447      792      41    horizon
      base_logreg     ALL      1d validation  0.476190   0.500000 0.409091 0.450000 0.534091 0.619036      977      42    horizon
      base_logreg     ALL     30d validation  0.452381   0.523810 0.458333 0.488889 0.483796 0.614578      977      42      model
      base_logreg     ALL     30d validation  0.452381   0.523810 0.458333 0.488889 0.483796 0.614578      977      42    section
      base_logreg     ALL     30d validati